# Model A — Full Fine-tune
**Task**: 4-class diabetic retinopathy grading (R0 / R1 / R2 / R3A)  
**Backbone**: RETFound-DINOv2 ViT-Large, all layers unfrozen  
**Best checkpoint** is saved automatically when val AUROC improves.  
Test-set evaluation runs automatically at the end using that checkpoint.

## Cell 1 — Imports & working directory

In [ ]:
import os, shutil, argparse
from pathlib import Path
import torch

# Make sure we are in the project root so relative paths work
os.chdir(os.path.dirname(os.path.abspath('train_modelA_ft.ipynb')))
print('Working directory:', os.getcwd())

## Cell 2 — Configuration
Edit these variables to change any hyperparameter.

In [ ]:
# ── Task identity ──────────────────────────────────────────────────────────────
ADAPTATION   = 'finetune'          # 'finetune' = all layers unfrozen
MODEL        = 'RETFound_dinov2'
MODEL_ARCH   = 'retfound_dinov2'
FINETUNE     = 'RETFound_dinov2_meh'  # HuggingFace repo id for pretrained weights
DATASET      = 'modelA'
NUM_CLASS    = 4

# ── Paths ──────────────────────────────────────────────────────────────────────
TASK         = f'{MODEL_ARCH}_{DATASET}_{ADAPTATION}'
DATA_PATH    = f'image_trees/{DATASET}'
OUTPUT_DIR   = f'output_dir/{TASK}'
LOG_DIR      = f'output_logs/{TASK}'

# ── Training ───────────────────────────────────────────────────────────────────
EPOCHS       = 50
BATCH_SIZE   = 24     # smaller than LP (64) because full backprop needs more VRAM
INPUT_SIZE   = 224

# ── Class weights (inverse-frequency, min scaled to 1.0) ──────────────────────
# R0=1.00  R1=1.79  R2=9.53  R3A=15.68
CLASS_WEIGHTS = [1.0, 1.7851, 9.5294, 15.6774]

print(f'Task : {TASK}')
print(f'Data : {DATA_PATH}')
print(f'Output: {OUTPUT_DIR}')
print(f'Class weights: {CLASS_WEIGHTS}')

## Cell 3 — Clean previous outputs  
Skip this cell if you want to resume from an existing checkpoint.

In [ ]:
for d in [OUTPUT_DIR, LOG_DIR]:
    full = os.path.join(d, TASK)
    if os.path.exists(full):
        shutil.rmtree(full)
        print(f'Deleted: {full}')
    else:
        print(f'Nothing to delete: {full}')

## Cell 4 — Build args & criterion

In [ ]:
from main_finetune import get_args_parser, main

# Build args from the same parser used by the bash script — no argparse magic needed
parser = get_args_parser()
args = parser.parse_args([
    '--model',       MODEL,
    '--model_arch',  MODEL_ARCH,
    '--finetune',    FINETUNE,
    '--adaptation',  ADAPTATION,
    '--nb_classes',  str(NUM_CLASS),
    '--data_path',   DATA_PATH,
    '--input_size',  str(INPUT_SIZE),
    '--task',        TASK,
    '--output_dir',  OUTPUT_DIR,
    '--log_dir',     LOG_DIR,
    '--batch_size',  str(BATCH_SIZE),
    '--epochs',      str(EPOCHS),
    '--world_size',  '1',
    '--class_weights', *[str(w) for w in CLASS_WEIGHTS],
])

# Build the weighted loss
weight    = torch.tensor(args.class_weights, dtype=torch.float)
criterion = torch.nn.CrossEntropyLoss(weight=weight)

# Create output directories
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print('Args ready. Criterion:', criterion)
print('class_weights:', args.class_weights)

## Cell 5 — Run training  
This cell runs all 50 epochs and then evaluates on the test set using the best checkpoint.  
Output prints epoch-by-epoch — scroll down to follow progress.

In [ ]:
main(args, criterion)

## Cell 6 — Inspect results

In [ ]:
import pandas as pd
import numpy as np

val_csv  = os.path.join(OUTPUT_DIR, TASK, 'metrics_val.csv')
test_csv = os.path.join(OUTPUT_DIR, TASK, 'metrics_test.csv')

df_val  = pd.read_csv(val_csv)
df_test = pd.read_csv(test_csv)

best_i = df_val['roc_auc'].idxmax()

print(f'Validation — best epoch: {best_i}  AUROC: {df_val["roc_auc"][best_i]:.4f}')
print(f'  Accuracy : {df_val["accuracy"][best_i]:.4f}')
print(f'  Kappa    : {df_val["kappa"][best_i]:.4f}')
print(f'  Macro Sensitivity: {df_val["macro_sensitivity"][best_i]:.4f}')
print(f'  Macro Specificity: {df_val["macro_specificity"][best_i]:.4f}')
print()
print('Test set (best checkpoint):')
t = df_test.iloc[0]
print(f'  AUROC    : {t["roc_auc"]:.4f}')
print(f'  Accuracy : {t["accuracy"]:.4f}')
print(f'  Kappa    : {t["kappa"]:.4f}')
print(f'  Macro Sensitivity: {t["macro_sensitivity"]:.4f}')
print(f'  Macro Specificity: {t["macro_specificity"]:.4f}')
classes = ['R0', 'R1', 'R2', 'R3A']
print()
print('Per-class test sensitivity:')
for i, c in enumerate(classes):
    print(f'  {c}: {t[f"sensitivity_{i}"]:.4f}')
print('Per-class test specificity:')
for i, c in enumerate(classes):
    print(f'  {c}: {t[f"specificity_{i}"]:.4f}')

## Cell 7 — AUROC curve (quick check)

In [ ]:
import matplotlib.pyplot as plt

ep = np.arange(len(df_val))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ep, df_val['roc_auc'], color='#1B7B8A', lw=2)
axes[0].axvline(best_i, color='#1B7B8A', lw=1, ls='--', alpha=0.6)
axes[0].set_title('Val AUROC'); axes[0].set_xlabel('Epoch'); axes[0].grid(True, alpha=0.4)

axes[1].plot(ep, df_val['val_loss'], color='#1B7B8A', lw=2, label='Val')
axes[1].set_title('Val Loss');  axes[1].set_xlabel('Epoch'); axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()